# TERA data inventory and quality control

This notebook verifies the analysis inputs, participant counts, outcomes, missingness, and lagged feature availability. It does not modify the source data.


In [ ]:
from pathlib import Path
import os, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 20260819
random.seed(SEED); np.random.seed(SEED)
warnings.filterwarnings('ignore')
DATA_ROOT = Path('TERA Analysis')
PROJECT_ROOT = Path('TERA')
RESULTS = PROJECT_ROOT / 'results'; FIGURES = PROJECT_ROOT / 'figures'
RESULTS.mkdir(exist_ok=True); FIGURES.mkdir(exist_ok=True)
print('Data:', DATA_ROOT)
print('Outputs:', PROJECT_ROOT)


In [ ]:
files = sorted(p for p in DATA_ROOT.rglob('*') if p.is_file())
inventory = pd.DataFrame({'path':[str(p.relative_to(DATA_ROOT)) for p in files],
                          'extension':[p.suffix.lower() for p in files],
                          'size_bytes':[p.stat().st_size for p in files]})
display(inventory.groupby('extension').size().sort_values(ascending=False).rename('file_count'))
display(inventory[inventory.extension.isin(['.csv','.xlsx','.ipynb'])].head(100))
inventory.to_csv(RESULTS/'data_inventory.csv',index=False)


In [ ]:
daily_path=DATA_ROOT/'exploratory_analysis/tera_daily.csv'
weekly_path=DATA_ROOT/'exploratory_analysis/weekly/tera_weekly.csv'
edm_path=DATA_ROOT/'Data/EDM_raw_data.csv'
for label,path,target in [('Daily',daily_path,'withinrange'),('Weekly',weekly_path,'is_adherent_wk')]:
    d=pd.read_csv(path)
    print(f'{label}: {len(d):,} rows; {d.USUBJID.nunique()} participants')
    display(d[target].value_counts(dropna=False).rename('n').to_frame())
    print('Duplicate rows:',d.duplicated().sum(),'| Missing target:',d[target].isna().sum())
edm=pd.read_csv(edm_path)
print(f'Raw EDM: {len(edm):,} rows; {edm.USUBJID.nunique()} participants')


In [ ]:
daily=pd.read_csv(daily_path); weekly=pd.read_csv(weekly_path)
qc=pd.DataFrame({
 'dataset':['daily','weekly'], 'rows':[len(daily),len(weekly)],
 'participants':[daily.USUBJID.nunique(),weekly.USUBJID.nunique()],
 'missing_outcome':[daily.withinrange.isna().sum(),weekly.is_adherent_wk.isna().sum()]})
display(qc); qc.to_csv(RESULTS/'dataset_qc_summary.csv',index=False)
